# 7. Beliefs Main

This notebook walks through the main belief-update recovery protocol step by step, from canonical cache initialization to the final six-rung experiment.


## Reading Guide

The goal here is to make the stricter protocol auditable end to end:

1. load the canonical cache;
2. build or load the shared manifest;
3. inspect the hidden current-state labels;
4. trace one example and manually reconstruct its update target;
5. inspect the sibling context and tensorized sample;
6. expose the grouped train/test split and rung plan;
7. make the target, feature blocks, models, losses, and metrics explicit;
8. run the main experiment and inspect compression diagnostics.


## Main Protocol

The main protocol asks a more direct representation question than the MVP outcome task: can non-local market context recover the hidden current-state update of the target market, and can that signal be compressed?

Protocol summary:

1. the target market's local state is observed only at stale time `t-Δ`;
2. siblings from the same weak family are observed at context time `t`;
3. the target market's own current state at `t` is hidden from model inputs;
4. the primary target is `label_update_logit = logit(p_A(t)) - logit(p_A(t-Δ))`;
5. the ladder compares raw context, large embeddings, compact embeddings, corrupted context, and an oracle upper bound.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from polymarket_research.belief_updating import (
    AGG_CONTEXT_FEATURE_NAMES,
    CONTEXT_FEATURE_NAMES,
    MAIN_FLAT_FEATURE_NAMES,
    BeliefUpdatingDatasetBuilder,
    BeliefUpdatingMainConfig,
    BeliefUpdatingMainExperiment,
    BeliefUpdatingManifest,
    BeliefUpdatingPredictor,
    BeliefUpdatingSpec,
    BeliefUpdatingTorchDataset,
    DeepSetsEncoder,
)
from polymarket_research.data.canonical import CanonicalDataset
from polymarket_research.utils import setup_root


## Step 1. Initialize Paths and Load the Canonical Dataset


In [ ]:
REPO_ROOT = setup_root()
DATA_SOURCE = 'polymarket'  # or 'kalshi'
ARTEFACT_ROOT = REPO_ROOT / 'frozen_notebooks' / 'running_artefacts' / DATA_SOURCE
CANONICAL_CACHE_DIR = ARTEFACT_ROOT / 'canonical_dataset'
MANIFEST_CACHE_DIR = ARTEFACT_ROOT / 'belief_updating_manifest'

canonical = CanonicalDataset.from_parquet(CANONICAL_CACHE_DIR)
spec = BeliefUpdatingSpec(
    stale_horizon_hours=24,
    local_lookback_hours=24,
    context_lookback_hours=72,
    horizon_hours=24,
    family_context_limit=16,
    time_split_quantile=0.8,
)
protocol = BeliefUpdateRecoveryBuilder(canonical, spec).build()


In [ ]:
display(canonical.summary())
if canonical.download_status is not None:
    display(canonical.status().head())


## Step 2. Inspect the Canonical Tables

We inspect both the market metadata table and the underlying probability trajectories because the update target is derived from both stale and current target snapshots.


In [ ]:
display(canonical.markets[[
    'market_id', 'question', 'research_category', 'family_id', 'created_at', 'end_date', 'final_yes_probability', 'probability_rows'
]].head(10))

display(canonical.probabilities.head(10))


## Step 3. Build or Load the Shared Belief Manifest


In [ ]:
display(manifest.summary())
print("main flat features:", MAIN_FLAT_FEATURE_NAMES)
print("context features:", manifest.context_feature_names)


## Step 4. Inspect the Main-Protocol Labels

The shared manifest stores the hidden current state only for supervision and analysis. These columns are not leaked into the flat input features.


In [ ]:
examples = manifest.examples.copy()
display(
    examples[[
        "market_id", "family_id", "horizon_hours", "delta_hours_int", "stale_yes_probability",
        "target_current_probability", "target_stale_logit", "target_current_logit",
        "label_update_logit", "label_stale_error_abs", "future_24h_repricing_label"
    ]].head(10)
)

display(examples[MAIN_FLAT_FEATURE_NAMES + ["label_update_logit"]].describe().T.head(25))


## Step 5. Trace One Example and Reconstruct the Update Target Manually

We pick one example, compute the stale logit, current logit, and their difference by hand, then compare against the stored manifest label.


In [ ]:
trace_row = examples.sort_values(["n_siblings", "delta_hours_int", "horizon_hours"], ascending=[False, False, True]).iloc[0]
trace_market_id = str(trace_row["market_id"])
trace_context_time = pd.Timestamp(trace_row["context_time"])
trace_stale_time = pd.Timestamp(trace_row["stale_time"])
trace_family_id = str(trace_row["family_id"])

def safe_logit(p, eps=1e-6):
    p = float(np.clip(p, eps, 1 - eps))
    return float(np.log(p / (1 - p)))

stale_logit_manual = safe_logit(trace_row["stale_yes_probability"])
current_logit_manual = safe_logit(trace_row["target_current_probability"])
update_manual = current_logit_manual - stale_logit_manual

manual_check = pd.DataFrame([{
    "stale_prob": float(trace_row["stale_yes_probability"]),
    "current_prob": float(trace_row["target_current_probability"]),
    "stale_logit_manual": stale_logit_manual,
    "stale_logit_manifest": float(trace_row["target_stale_logit"]),
    "current_logit_manual": current_logit_manual,
    "current_logit_manifest": float(trace_row["target_current_logit"]),
    "update_manual": update_manual,
    "update_manifest": float(trace_row["label_update_logit"]),
}])

display(trace_row.to_frame(name="value").head(30))
display(manual_check)


## Step 6. Inspect the Target Trajectory and Sibling Context at Context Time


In [ ]:
target_panel = canonical.probabilities.loc[canonical.probabilities["market_id"].astype(str) == trace_market_id].copy()
target_window = target_panel.loc[
    target_panel["timestamp_utc"].between(trace_stale_time - pd.Timedelta(hours=36), trace_context_time + pd.Timedelta(hours=1))
].copy()
display(target_window.tail(20))

context_rows = manifest.context_snapshots.loc[
    (manifest.context_snapshots["family_id"].astype(str) == trace_family_id)
    & (manifest.context_snapshots["snapshot_time"] == trace_context_time)
    & (manifest.context_snapshots["market_id"].astype(str) != trace_market_id)
].copy()
context_rows["ctx_prob_vs_target_stale"] = context_rows["ctx_yes_probability"] - float(trace_row["stale_yes_probability"])
display(context_rows[[
    "market_id", "ctx_yes_probability", "ctx_confidence_margin", "ctx_prob_vs_target_stale",
    "ctx_lookback_24h_change", "ctx_lookback_24h_volatility", "ctx_lookback_24h_log_trade"
]].head(20))

display(trace_row[manifest.global_feature_names].to_frame(name="value"))


## Step 7. Inspect the Tensorized Main-Protocol Sample

The main protocol uses the same set encoder substrate as the MVP, but with a different flat feature block and a different label column.


In [ ]:
torch_dataset = BeliefUpdatingTorchDataset(
    manifest,
    indices=[int(trace_row.name)],
    max_context_size=16,
    flat_feature_names=MAIN_FLAT_FEATURE_NAMES,
    label_col="label_update_logit",
)
sample = torch_dataset[0]
print("flat feature vector shape:", tuple(sample["flat"].shape))
print("context matrix shape:", tuple(sample["context"].shape))
print("valid siblings:", int(sample["mask"].sum().item()))
print("update label:", float(sample["label"].item()))

display(pd.DataFrame(sample["context"][sample["mask"]].numpy(), columns=manifest.context_feature_names).head())


## Step 8. Define the Train/Test Split Explicitly

Before training, expose the grouped out-of-time split, the target column, and the rung plan directly from the public experiment interface.


In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

config = BeliefUpdatingMainConfig(
    test_fraction=0.25,
    n_epochs=10,
    batch_size=128,
    large_encoder_output_dim=128,
    compact_encoder_output_dim=32,
    device=device,
    use_global_features=True,
    log_every_epochs=1,
    show_epoch_progress=True,
)

experiment = BeliefUpdatingMainExperiment(manifest, config=config)
artifact_bundle = experiment.artifacts(preview_rows=10)
train_df, test_df = experiment._split()

display(artifact_bundle.protocol_summary)
display(artifact_bundle.split_summary)
display(artifact_bundle.rung_plan)
display(artifact_bundle.train_preview)
display(artifact_bundle.test_preview)


## Step 9. Make Features and Target Explicit

The main protocol predicts the hidden current-state update in logit space. Show the flat features, aggregated raw-context features, and per-sibling context features before any model fitting.


In [ ]:
target_col = artifact_bundle.protocol_summary.iloc[0]["target_column"]
feature_blocks = experiment.feature_blocks()

display(artifact_bundle.feature_blocks)

flat_feature_cols = list(feature_blocks["flat_main_features"])
raw_context_feature_cols = [c for c in AGG_CONTEXT_FEATURE_NAMES if c in manifest.examples.columns]
context_feature_cols = list(feature_blocks["context_per_sibling"])

print("target column:", target_col)
print("flat main feature columns:", flat_feature_cols)
print("aggregated raw-context columns:", raw_context_feature_cols)
print("context per-sibling feature columns:", context_feature_cols)

display(manifest.examples[flat_feature_cols + ["target_current_probability", target_col]].head(5))


## Step 10. Make the Models Explicit

The first two rungs use a tabular regressor. The compression rungs use a DeepSets encoder plus a prediction head. The oracle rung simply reads the hidden current state for an upper bound.


In [ ]:
display(artifact_bundle.model_registry)

gbm_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", HistGradientBoostingRegressor(
        max_iter=config.gbm_max_iter,
        max_depth=4,
        learning_rate=0.05,
        random_state=config.random_state,
    )),
])

large_encoder = DeepSetsEncoder(
    input_dim=len(CONTEXT_FEATURE_NAMES),
    hidden_dim=config.encoder_hidden_dim,
    output_dim=config.large_encoder_output_dim,
)
compact_encoder = DeepSetsEncoder(
    input_dim=len(CONTEXT_FEATURE_NAMES),
    hidden_dim=config.encoder_hidden_dim,
    output_dim=config.compact_encoder_output_dim,
)
predictor = BeliefUpdatingPredictor(
    flat_dim=len(flat_feature_cols),
    context_dim=config.compact_encoder_output_dim,
)

print(gbm_pipeline)
print()
print("Large encoder:")
print(large_encoder)
print()
print("Compact encoder:")
print(compact_encoder)
print()
print("Prediction head:")
print(predictor)
print()
print("Oracle rung: uses the true hidden current state and does not fit a deployable model.")


## Step 11. Make Losses and Metrics Explicit

The PyTorch rungs optimize squared error on `label_update_logit`. Reporting happens in both update space and reconstructed current-probability space.


In [ ]:
display(artifact_bundle.objective_registry)

criterion = nn.MSELoss()
print("PyTorch loss for the embedding rungs:", criterion)

update_true = test_df[target_col].to_numpy(dtype=float)
zero_update_pred = np.zeros(len(test_df), dtype=float)
zero_update_current_prob = test_df["stale_yes_probability"].to_numpy(dtype=float)
current_prob_true = test_df["target_current_probability"].to_numpy(dtype=float)

zero_update_err = zero_update_pred - update_true
zero_prob_err = zero_update_current_prob - current_prob_true
ss_res = float(np.sum(zero_update_err ** 2))
ss_tot = float(np.sum((update_true - np.mean(update_true)) ** 2))
zero_update_r2 = 1.0 - (ss_res / ss_tot) if ss_tot > 1e-12 else float("nan")

print("example zero-update MAE:", round(float(np.mean(np.abs(zero_update_err))), 5))
print("example zero-update RMSE:", round(float(np.sqrt(np.mean(zero_update_err ** 2))), 5))
print("example zero-update R^2:", round(float(zero_update_r2), 5))
print("example stale-probability Brier:", round(float(np.mean(zero_prob_err ** 2)), 5))
print("example stale-probability MAE:", round(float(np.mean(np.abs(zero_prob_err))), 5))


## Step 12. Run the Main Belief-Update Experiment

The runner now prints a protocol summary, per-rung headers, and epoch-by-epoch progress for the set-encoder rungs.


In [ ]:
results = experiment.run(verbose=True)
display(results.to_dataframe())
display(results.compression_summary())

for rung_name, history in results.training_histories().items():
    print(f"training history: {rung_name}")
    display(history.tail())


## Step 13. Visualize the Main-Protocol Results


In [ ]:
frame = results.to_dataframe()
histories = results.training_histories()
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

frame.plot(x="rung", y="update_rmse", kind="bar", legend=False, ax=axes[0], color="#B45309")
axes[0].set_title("Main rung update RMSE")
axes[0].set_xlabel("Rung")
axes[0].set_ylabel("Update RMSE")
axes[0].tick_params(axis="x", rotation=35)

frame.plot(x="rung", y="current_prob_brier", kind="bar", legend=False, ax=axes[1], color="#0F766E")
axes[1].set_title("Reconstructed current-probability Brier")
axes[1].set_xlabel("Rung")
axes[1].set_ylabel("Brier")
axes[1].tick_params(axis="x", rotation=35)

if histories:
    for rung_name, history in histories.items():
        axes[2].plot(history["epoch"], history["mean_train_loss"], marker="o", label=rung_name)
    axes[2].legend(frameon=False)
    axes[2].set_title("PyTorch training traces")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Mean train loss")
else:
    axes[2].text(0.5, 0.5, "No training histories recorded", ha="center", va="center")
    axes[2].set_axis_off()

plt.tight_layout()


## What to Look For

The main compression claim is supported when:

- `stale_plus_raw` improves over `stale_only`;
- `stale_plus_large_embedding` improves further or at least remains competitive;
- `stale_plus_compact_embedding` stays close to the large embedding;
- `stale_plus_corrupted` falls back toward stale-only;
- `current_local_oracle` remains a clear upper bound.

That pattern means the hidden current-state update is recoverable from non-local context and survives compression into a compact representation.
